# Exercise 4 — Detect Resources Missing from Some Cluster Nodes

## Objective

A web API is load-balanced across multiple nodes. Each node reports a dictionary where:

- the **key** is a requested resource;
- the **value** is the number of requests handled by that node.

We need to find resources that were requested on **some, but not all, nodes**.

For each such resource, return the request count for every node in node order, using `0` when the resource does not appear on a node.

The exercise describes exactly three nodes, but the enhanced implementation later in this notebook supports any number of nodes.

## Important Note About the Required Output

The exercise text says that each result value should be a **list**, but the example output uses **tuples**:

```python
'employee': (5000, 0, 0)
```

This notebook uses tuples by default because they match the supplied expected output. An enhanced version later allows either tuples or lists.

## Example Node Data

In [1]:
n1 = {
    'employees': 100,
    'employee': 5000,
    'users': 10,
    'user': 100,
}

n2 = {
    'employees': 250,
    'users': 23,
    'user': 230,
}

n3 = {
    'employees': 150,
    'users': 4,
    'login': 1000,
}

The expected result is:

```python
{
    'employee': (5000, 0, 0),
    'user': (100, 230, 0),
    'login': (0, 0, 1000),
}
```

`employees` and `users` do **not** appear because they are present on all three nodes.

---

# Understanding the Set Logic

We need resources that exist in **at least one node**, but do **not** exist in every node.

Mathematically:

```text
problematic resources = union of keys - intersection of keys
```

For three nodes:

```python
all_resources = n1.keys() | n2.keys() | n3.keys()
common_resources = n1.keys() & n2.keys() & n3.keys()

problematic_resources = all_resources - common_resources
```

This corresponds exactly to the load-balancing condition described in the exercise.

In [2]:
all_resources = n1.keys() | n2.keys() | n3.keys()
common_resources = n1.keys() & n2.keys() & n3.keys()
problematic_resources = all_resources - common_resources

print('All resources:       ', all_resources)
print('On every node:       ', common_resources)
print('Missing somewhere:   ', problematic_resources)

All resources:        {'employees', 'user', 'login', 'users', 'employee'}
On every node:        {'employees', 'users'}
Missing somewhere:    {'user', 'login', 'employee'}


---

# Core Exercise Solution

Using the union-minus-intersection idea, the function can be written very compactly.

In [3]:
def find_unbalanced_resources(n1, n2, n3):
    """Return resources present on some, but not all, three nodes."""
    all_keys = n1.keys() | n2.keys() | n3.keys()
    common_keys = n1.keys() & n2.keys() & n3.keys()
    target_keys = all_keys - common_keys

    return {
        key: (n1.get(key, 0), n2.get(key, 0), n3.get(key, 0))
        for key in target_keys
    }

In [4]:
result = find_unbalanced_resources(n1, n2, n3)
print(result)

{'user': (100, 230, 0), 'login': (0, 0, 1000), 'employee': (5000, 0, 0)}


Because `target_keys` is a set, key ordering is not part of this basic solution's contract.

The values, however, always correspond to the nodes in the correct order:

```text
(node 1, node 2, node 3)
```

---

# Verify the Exercise Result

In [5]:
expected = {
    'employee': (5000, 0, 0),
    'user': (100, 230, 0),
    'login': (0, 0, 1000),
}

assert find_unbalanced_resources(n1, n2, n3) == expected

print('Exercise requirement test passed.')

Exercise requirement test passed.


---

# Why `dict.get()` Is Useful Here

For a resource that may be missing from a node, this expression:

```python
node.get(resource, 0)
```

means:

- return the recorded count when the resource exists;
- otherwise return `0`.

This is cleaner than repeatedly writing membership checks such as:

```python
node[resource] if resource in node else 0
```

---

# Recommended Generalized Solution

The original exercise assumes exactly three nodes, but real clusters may contain any number of nodes.

The following implementation generalizes the algorithm to **N nodes**.

A resource is returned if it occurs in at least one node but not every node.

In [6]:
from collections.abc import Mapping
from numbers import Real
from typing import Hashable


def find_partial_resources(
    *nodes: Mapping[Hashable, Real],
) -> dict[Hashable, tuple[Real, ...]]:
    """Return resources present on some, but not all, nodes.

    Parameters
    ----------
    *nodes:
        Two or more mappings of resource -> request count.

    Returns
    -------
    dict
        Each returned key is present on at least one node but absent
        from at least one other node.

        Each value is a tuple containing request counts in node order.
        A missing resource contributes 0.

    Raises
    ------
    ValueError
        If fewer than two nodes are supplied.
    """
    if len(nodes) < 2:
        raise ValueError('At least two nodes are required.')

    all_keys = set().union(*(node.keys() for node in nodes))
    common_keys = set(nodes[0]).intersection(
        *(node.keys() for node in nodes[1:])
    )

    partial_keys = all_keys - common_keys

    return {
        key: tuple(node.get(key, 0) for node in nodes)
        for key in partial_keys
    }

In [7]:
result = find_partial_resources(n1, n2, n3)
print(result)

{'user': (100, 230, 0), 'login': (0, 0, 1000), 'employee': (5000, 0, 0)}


---

# Deterministic Output Ordering

Set operations are ideal for determining membership, but applications often benefit from reproducible output.

A convenient policy is to sort resource names alphabetically.

In [8]:
def find_partial_resources_sorted(*nodes):
    """Return partially distributed resources in deterministic key order."""
    if len(nodes) < 2:
        raise ValueError('At least two nodes are required.')

    all_keys = set().union(*(node.keys() for node in nodes))
    common_keys = set(nodes[0]).intersection(
        *(node.keys() for node in nodes[1:])
    )

    partial_keys = all_keys - common_keys

    return {
        key: tuple(node.get(key, 0) for node in nodes)
        for key in sorted(partial_keys, key=str)
    }

In [9]:
sorted_result = find_partial_resources_sorted(n1, n2, n3)
print(sorted_result)

{'employee': (5000, 0, 0), 'login': (0, 0, 1000), 'user': (100, 230, 0)}


Expected deterministic ordering:

```text
{'employee': (5000, 0, 0), 'login': (0, 0, 1000), 'user': (100, 230, 0)}
```

---

# Production-Style Implementation

The next version adds:

- support for any number of nodes;
- input validation;
- validation of request counts;
- deterministic ordering;
- configurable tuple/list output;
- optional exclusion of resources with zero total traffic;
- non-mutating behavior.

In [10]:
def analyze_partial_distribution(
    *nodes: Mapping[Hashable, Real],
    as_lists: bool = False,
    include_zero_total: bool = False,
    sort_keys: bool = True,
) -> dict:
    """Find resources that are not represented on every node.

    Parameters
    ----------
    *nodes:
        Two or more resource-count mappings.
    as_lists:
        If True, return count sequences as lists instead of tuples.
    include_zero_total:
        If False, omit resources whose combined count is zero.
    sort_keys:
        If True, return resources in deterministic order.

    Returns
    -------
    dict
        Mapping of partially distributed resources to per-node counts.

    Raises
    ------
    ValueError
        If fewer than two nodes are supplied or a count is negative.
    TypeError
        If an input is not a mapping or a count is not numeric.
    """
    if len(nodes) < 2:
        raise ValueError('At least two nodes are required.')

    for node_index, node in enumerate(nodes, start=1):
        if not isinstance(node, Mapping):
            raise TypeError(
                f'Node {node_index} must be a mapping; '
                f'got {type(node).__name__}.'
            )

        for resource, count in node.items():
            if isinstance(count, bool) or not isinstance(count, Real):
                raise TypeError(
                    f'Count for {resource!r} on node {node_index} '
                    f'must be numeric.'
                )

            if count < 0:
                raise ValueError(
                    f'Count for {resource!r} on node {node_index} '
                    f'cannot be negative.'
                )

    all_keys = set().union(*(node.keys() for node in nodes))
    common_keys = set(nodes[0]).intersection(
        *(node.keys() for node in nodes[1:])
    )

    partial_keys = all_keys - common_keys

    if sort_keys:
        partial_keys = sorted(partial_keys, key=str)

    result = {}

    for resource in partial_keys:
        counts = tuple(node.get(resource, 0) for node in nodes)

        if not include_zero_total and sum(counts) == 0:
            continue

        result[resource] = list(counts) if as_lists else counts

    return result

The loop in this production-style version is intentional: validation and configurable result construction make explicit logic easier to read than forcing every operation into one comprehension.

## Run the Enhanced Version

In [11]:
result = analyze_partial_distribution(n1, n2, n3)
print(result)

{'employee': (5000, 0, 0), 'login': (0, 0, 1000), 'user': (100, 230, 0)}


Expected output:

```text
{'employee': (5000, 0, 0), 'login': (0, 0, 1000), 'user': (100, 230, 0)}
```

---

# Optional List Output

If the literal wording of the exercise is preferred instead of its tuple-based example, set `as_lists=True`.

In [12]:
list_result = analyze_partial_distribution(
    n1,
    n2,
    n3,
    as_lists=True,
)

print(list_result)

{'employee': [5000, 0, 0], 'login': [0, 0, 1000], 'user': [100, 230, 0]}


Expected output:

```text
{'employee': [5000, 0, 0], 'login': [0, 0, 1000], 'user': [100, 230, 0]}
```

---

# Added Value: Four-Node Example

The generalized solution works without modification if another API node is added.

In [13]:
n4 = {
    'employees': 175,
    'users': 8,
    'login': 300,
    'reports': 75,
}

four_node_result = analyze_partial_distribution(
    n1,
    n2,
    n3,
    n4,
)

print(four_node_result)

{'employee': (5000, 0, 0, 0), 'login': (0, 0, 1000, 300), 'reports': (0, 0, 0, 75), 'user': (100, 230, 0, 0)}


Every returned value now contains four entries, corresponding exactly to:

```text
(n1, n2, n3, n4)
```

---

# Added Value: Coverage Diagnostics

Returning raw counts identifies a problem, but operational monitoring often benefits from knowing **how many nodes** saw each resource.

The helper below computes a coverage report.

In [14]:
def resource_coverage(*nodes):
    """Return node-coverage diagnostics for every resource."""
    if len(nodes) < 1:
        return {}

    all_keys = set().union(*(node.keys() for node in nodes))
    node_count = len(nodes)

    return {
        resource: {
            'nodes_present': sum(resource in node for node in nodes),
            'nodes_total': node_count,
            'coverage_ratio': sum(resource in node for node in nodes) / node_count,
            'counts': tuple(node.get(resource, 0) for node in nodes),
        }
        for resource in sorted(all_keys, key=str)
    }

In [15]:
coverage = resource_coverage(n1, n2, n3)

for resource, details in coverage.items():
    print(resource, '->', details)

employee -> {'nodes_present': 1, 'nodes_total': 3, 'coverage_ratio': 0.3333333333333333, 'counts': (5000, 0, 0)}
employees -> {'nodes_present': 3, 'nodes_total': 3, 'coverage_ratio': 1.0, 'counts': (100, 250, 150)}
login -> {'nodes_present': 1, 'nodes_total': 3, 'coverage_ratio': 0.3333333333333333, 'counts': (0, 0, 1000)}
user -> {'nodes_present': 2, 'nodes_total': 3, 'coverage_ratio': 0.6666666666666666, 'counts': (100, 230, 0)}
users -> {'nodes_present': 3, 'nodes_total': 3, 'coverage_ratio': 1.0, 'counts': (10, 23, 4)}


For example:

- `employees` has coverage `3 / 3 = 1.0`;
- `user` has coverage `2 / 3`;
- `employee` has coverage `1 / 3`;
- `login` has coverage `1 / 3`.

This makes the severity of distribution gaps easier to inspect.

---

# Added Value: Detailed Imbalance Report

For diagnostics, we can report not only the counts but also exactly which node numbers are missing a resource.

In [16]:
def imbalance_report(*nodes):
    """Return detailed diagnostics for resources missing from any node."""
    partial = analyze_partial_distribution(*nodes)
    node_count = len(nodes)

    return {
        resource: {
            'counts': counts,
            'present_on': tuple(
                index
                for index, node in enumerate(nodes, start=1)
                if resource in node
            ),
            'missing_from': tuple(
                index
                for index, node in enumerate(nodes, start=1)
                if resource not in node
            ),
            'coverage': sum(resource in node for node in nodes) / node_count,
            'total_requests': sum(counts),
        }
        for resource, counts in partial.items()
    }

In [17]:
report = imbalance_report(n1, n2, n3)

for resource, details in report.items():
    print(f'{resource}: {details}')

employee: {'counts': (5000, 0, 0), 'present_on': (1,), 'missing_from': (2, 3), 'coverage': 0.3333333333333333, 'total_requests': 5000}
login: {'counts': (0, 0, 1000), 'present_on': (3,), 'missing_from': (1, 2), 'coverage': 0.3333333333333333, 'total_requests': 1000}
user: {'counts': (100, 230, 0), 'present_on': (1, 2), 'missing_from': (3,), 'coverage': 0.6666666666666666, 'total_requests': 330}


This is closer to the information an operations or SRE team might actually consume when diagnosing routing anomalies.

---

# Exercise Requirement Tests

In [18]:
expected = {
    'employee': (5000, 0, 0),
    'login': (0, 0, 1000),
    'user': (100, 230, 0),
}

assert analyze_partial_distribution(n1, n2, n3) == expected

assert analyze_partial_distribution(
    n1,
    n2,
    n3,
    as_lists=True,
) == {
    'employee': [5000, 0, 0],
    'login': [0, 0, 1000],
    'user': [100, 230, 0],
}

print('Exercise requirement tests passed.')

Exercise requirement tests passed.


---

# Edge-Case Tests

In [19]:
# All nodes empty
assert analyze_partial_distribution({}, {}) == {}

# Same resources on every node -> nothing is partially distributed
assert analyze_partial_distribution(
    {'a': 1, 'b': 2},
    {'a': 10, 'b': 20},
) == {}

# Resource exists on one node only
assert analyze_partial_distribution(
    {'a': 10},
    {},
    {},
) == {
    'a': (10, 0, 0),
}

# Different unique resource on every node
assert analyze_partial_distribution(
    {'a': 1},
    {'b': 2},
    {'c': 3},
) == {
    'a': (1, 0, 0),
    'b': (0, 2, 0),
    'c': (0, 0, 3),
}

# A zero count can still represent an explicitly present resource
assert analyze_partial_distribution(
    {'a': 0, 'common': 1},
    {'common': 2},
) == {}

# Why is 'a' omitted above? It is present on only one node but has a
# combined request count of zero. Set include_zero_total=True to retain it.
assert analyze_partial_distribution(
    {'a': 0, 'common': 1},
    {'common': 2},
    include_zero_total=True,
) == {
    'a': (0, 0),
}

print('Edge-case tests passed.')

Edge-case tests passed.


---

# Validation Tests

In [20]:
def assert_raises(expected_exception, function, *args, **kwargs):
    """Verify that a function raises the expected exception."""
    try:
        function(*args, **kwargs)
    except expected_exception:
        return
    except Exception as exc:
        raise AssertionError(
            f'Expected {expected_exception.__name__}, '
            f'but got {type(exc).__name__}.'
        ) from exc

    raise AssertionError(
        f'Expected {expected_exception.__name__} to be raised.'
    )

In [21]:
# Too few nodes
assert_raises(
    ValueError,
    analyze_partial_distribution,
    {'a': 1},
)

# Non-mapping node
assert_raises(
    TypeError,
    analyze_partial_distribution,
    {'a': 1},
    ['not', 'a', 'mapping'],
)

# Non-numeric count
assert_raises(
    TypeError,
    analyze_partial_distribution,
    {'a': '100'},
    {},
)

# Boolean counts are deliberately rejected
assert_raises(
    TypeError,
    analyze_partial_distribution,
    {'a': True},
    {},
)

# Negative request count
assert_raises(
    ValueError,
    analyze_partial_distribution,
    {'a': -10},
    {},
)

print('Validation tests passed.')

Validation tests passed.


---

# Verify Inputs Are Not Modified

In [22]:
n1_before = n1.copy()
n2_before = n2.copy()
n3_before = n3.copy()

_ = analyze_partial_distribution(n1, n2, n3)

assert n1 == n1_before
assert n2 == n2_before
assert n3 == n3_before

print('Input dictionaries were not modified.')

Input dictionaries were not modified.


---

# Important Semantic Detail: Missing vs Zero

There is an important distinction between:

```python
{}
```

and:

```python
{'login': 0}
```

In the first case, `login` is **absent** from the node's data.

In the second case, `login` is explicitly **present**, but its recorded count is zero.

The load-balancer condition in this exercise is based on **key presence**, not whether the count is greater than zero.

Therefore the set calculations correctly use dictionary keys rather than values.

---

# Why Not Compare Counts?

The goal is not to determine whether every node handled the **same number** of requests.

For example:

```python
'employees': (100, 250, 150)
```

has very different counts, but `employees` exists on all three nodes, so according to the exercise it is **not** included in the result.

This exercise detects **missing routing coverage**, not statistical traffic imbalance.

A production load-balancer monitor would likely perform both checks separately.

---

# Added Value: Detect Suspicious Traffic Imbalance

A resource can occur on every node and still be distributed very unevenly.

The following helper provides a simple imbalance metric based on each node's share of requests.

In [23]:
def traffic_distribution(*nodes):
    """Return per-node traffic shares for each resource."""
    if not nodes:
        return {}

    all_resources = set().union(*(node.keys() for node in nodes))
    result = {}

    for resource in sorted(all_resources, key=str):
        counts = tuple(node.get(resource, 0) for node in nodes)
        total = sum(counts)

        shares = (
            tuple(count / total for count in counts)
            if total
            else tuple(0.0 for _ in counts)
        )

        result[resource] = {
            'counts': counts,
            'total': total,
            'shares': shares,
            'spread': max(shares) - min(shares) if shares else 0.0,
        }

    return result

In [24]:
distribution = traffic_distribution(n1, n2, n3)

for resource, details in distribution.items():
    print(resource, '->', details)

employee -> {'counts': (5000, 0, 0), 'total': 5000, 'shares': (1.0, 0.0, 0.0), 'spread': 1.0}
employees -> {'counts': (100, 250, 150), 'total': 500, 'shares': (0.2, 0.5, 0.3), 'spread': 0.3}
login -> {'counts': (0, 0, 1000), 'total': 1000, 'shares': (0.0, 0.0, 1.0), 'spread': 1.0}
user -> {'counts': (100, 230, 0), 'total': 330, 'shares': (0.30303030303030304, 0.696969696969697, 0.0), 'spread': 0.696969696969697}
users -> {'counts': (10, 23, 4), 'total': 37, 'shares': (0.2702702702702703, 0.6216216216216216, 0.10810810810810811), 'spread': 0.5135135135135135}


The `spread` value ranges from `0` toward `1`:

- a small spread means traffic shares are relatively similar;
- a large spread means traffic is concentrated on some nodes.

This is not required by the exercise, but it illustrates the difference between **resource coverage** and **traffic balance**.

---

# Complexity Analysis

Let:

- **S** = number of nodes;
- **N** = total number of key/value entries across all node dictionaries;
- **U** = number of unique resources;
- **P** = number of resources that are present on some but not all nodes.

## Union and intersection

Building the union and intersection requires processing the node keys, giving approximately:

**Time:** `O(N)` average-case

## Constructing result values

For each of the `P` problematic resources, the algorithm looks up the resource on all `S` nodes:

```text
O(P × S)
```

Since dictionary lookup is `O(1)` average-case, total expected time is:

**Time:** `O(N + P × S)`

If deterministic sorting is requested, add:

**Sorting:** `O(P log P)`

The main auxiliary structures contain sets of resource keys:

**Space:** `O(U + P × S)` including the returned count tuples.

---

# Practical Interpretation

The algorithm can be viewed as a distributed consistency check:

```text
Node 1 logs ─┐
Node 2 logs ─┼──> key coverage analysis ──> missing-node report
Node 3 logs ─┘
```

The same pattern can help identify inconsistencies in:

- API routing;
- distributed caches;
- replicated datasets;
- event consumers;
- shard coverage;
- regional traffic;
- microservice instances;
- distributed monitoring data.

---

# Final Answer

For the exact three-node exercise, the essential Pythonic solution is:

```python
def find_unbalanced_resources(n1, n2, n3):
    all_keys = n1.keys() | n2.keys() | n3.keys()
    common_keys = n1.keys() & n2.keys() & n3.keys()

    return {
        key: (n1.get(key, 0), n2.get(key, 0), n3.get(key, 0))
        for key in all_keys - common_keys
    }
```

The key insight is:

```text
(union of all keys) - (intersection of all keys)
```

That isolates exactly the resources that occur on **some, but not all**, nodes.

## Summary

Key ideas demonstrated in this exercise:

- dictionary key views;
- set union with `|`;
- set intersection with `&`;
- set difference with `-`;
- `dict.get(key, 0)` for missing resources;
- dictionary comprehensions;
- arbitrary numbers of cluster nodes;
- deterministic output;
- tuple/list output options;
- input validation;
- resource-coverage diagnostics;
- missing-node diagnostics;
- traffic-distribution analysis;
- distinction between missing keys and zero counts;
- non-mutating API design;
- complexity analysis.